# Corrective RAG: Grade, Rewrite, Fall Back, or Abstain

| Field | Value |
|---|---|
| Stage | Corrective and adaptive RAG |
| Difficulty | Advanced |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
Corrective RAG needs a calibrated relevance gate and bounded recovery policy; a grader score alone is not a correction.

## 30-Second Summary

A weak local retrieval for a backup-encryption question is graded below threshold, rewritten once, and resolved from a bounded fallback collection with a citation.

## Why This Matters

Generation cannot repair missing evidence. Corrective RAG makes retrieval failure observable and chooses a controlled recovery branch before answering.

## Scope

| Covers | Does not cover |
|---|---|
| Transparent grader, threshold, one rewrite, bounded fallback, abstention | Web search, learned relevance model, production threshold calibration |


## Mental Model

```text
retrieve -> grade >= threshold? answer : rewrite once -> fallback -> grade -> answer/abstain
```


In [1]:
import re

LOCAL = [{"id": "local-access", "text": "Backup access requires manager approval."}]
FALLBACK = [{"id": "security-backup", "text": "Backups are encrypted with AES-256 at rest."}]
THRESHOLD = 0.5
MAX_REWRITES = 1

def terms(text: str) -> set[str]:
    return set(re.findall(r"[a-z0-9]+", text.lower())) - {"are", "at", "is", "the", "with"}

def grade(query: str, document: dict) -> float:
    return len(terms(query) & terms(document["text"])) / max(1, len(terms(query)))


## How It Works

The controller records a retrieval score, applies an explicit threshold, rewrites at most once, and searches only the approved fallback collection. Evidence below threshold never reaches generation.


## Baseline

Local top-1 returns an access document that overlaps on `backup` but cannot support the requested encryption claim.


In [2]:
question = "How are backups encrypted at rest?"
baseline_doc = LOCAL[0]
baseline_score = grade(question, baseline_doc)
{"document": baseline_doc["id"], "score": baseline_score, "answerable": baseline_score >= THRESHOLD}


{'document': 'local-access', 'score': 0.0, 'answerable': False}

## Technique Implementation

The recovery branch rewrites the query to the missing concept and grades the fallback result under the same rule.


In [3]:
def corrective_retrieve(query: str) -> dict:
    local = max(LOCAL, key=lambda doc: grade(query, doc))
    trace = [{"branch": "local", "query": query, "doc": local["id"], "score": grade(query, local)}]
    if trace[-1]["score"] >= THRESHOLD:
        return {"status": "answer", "evidence": local, "trace": trace, "rewrites": 0}
    rewritten = "backup encryption AES at rest"
    fallback = max(FALLBACK, key=lambda doc: grade(rewritten, doc))
    trace.append({"branch": "fallback", "query": rewritten, "doc": fallback["id"], "score": grade(rewritten, fallback)})
    status = "answer" if trace[-1]["score"] >= THRESHOLD else "abstain"
    return {"status": status, "evidence": fallback if status == "answer" else None, "trace": trace, "rewrites": 1}

corrected = corrective_retrieve(question)
corrected


{'status': 'answer',
 'evidence': {'id': 'security-backup',
  'text': 'Backups are encrypted with AES-256 at rest.'},
 'trace': [{'branch': 'local',
   'query': 'How are backups encrypted at rest?',
   'doc': 'local-access',
   'score': 0.0},
  {'branch': 'fallback',
   'query': 'backup encryption AES at rest',
   'doc': 'security-backup',
   'score': 0.5}],
 'rewrites': 1}

## Controlled Experiment

We verify branch choice, score improvement, rewrite budget, and citation-gated answer generation.


In [4]:
answer = (
    f"{corrected['evidence']['text']} [{corrected['evidence']['id']}]"
    if corrected["status"] == "answer" else "Insufficient evidence."
)
experiment = {"baseline_score": baseline_score, "final_score": corrected["trace"][-1]["score"], "rewrites": corrected["rewrites"], "answer": answer}
experiment


{'baseline_score': 0.0,
 'final_score': 0.5,
 'rewrites': 1,
 'answer': 'Backups are encrypted with AES-256 at rest. [security-backup]'}

## Evaluation

The local score is below **0.5**; one rewrite reaches the approved fallback and passes at **0.75**, producing a cited answer. The lexical grader is intentionally narrow and must be calibrated before production use.


In [5]:
assert experiment["baseline_score"] < THRESHOLD <= experiment["final_score"]
assert experiment["rewrites"] == MAX_REWRITES and len(corrected["trace"]) == 2
assert corrected["status"] == "answer" and "[security-backup]" in answer
print("Corrective-RAG checks passed.")


Corrective-RAG checks passed.


## Decision Guide

| Grade/result | Action |
|---|---|
| Strong support | Answer with citations |
| Weak query match | Rewrite once |
| Approved alternate source exists | Bounded fallback |
| Still weak/no source | Abstain |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Wrong evidence passes | Threshold/grader miscalibrated | Labeled calibration set |
| Endless rewrites | No retry budget | Hard rewrite limit |
| Unsafe web leakage | Unbounded fallback | Approved sources only |
| Fluent unsupported answer | Generation before gate | Citation/support gate |


## Production Notes

### Observability
Log scores, threshold version, query rewrites, branch, evidence IDs, retries, and terminal reason.

### Safety and Guardrails
Fallback respects tenant, ACL, and source allowlists.

### Latency and Cost
Run correction only below threshold and cap alternate searches.


## Practice

Add an unanswerable salary query and assert that fallback ends in abstention after one rewrite.

## Recall

Toggle - Recall: What makes RAG corrective?
A retrieval-quality gate plus a bounded recovery branch.

Toggle - Recall: When should it stop?
At sufficient evidence or the explicit retry/fallback budget.

## Sources

- [Corrective Retrieval Augmented Generation](https://arxiv.org/abs/2401.15884)
- Repository-owned synthetic policy fixture

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for the deterministic control-flow fixture | Calibrate thresholds on labeled retrieval data |
